In [0]:
%run ../_shared/00_project_config

In [0]:
%run ../_shared/01_spark_utils

In [0]:
from pyspark.sql import functions as F

In [0]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide paths, schemas, and canonical league mappings are loaded from
# 00_project_config.
#
# FOOTBALL_DATA_ORG_COMPETITIONS maps football-data.org competition codes to
# canonical platform league identifiers:
#
#   PL  -> premier_league
#   PD  -> la_liga
#   BL1 -> bundesliga
#   SA  -> serie_a
#   FL1 -> ligue_one
#
# Bronze preserves the nested JSON source structures. Flattening and explicit
# analytical typing are deferred to Silver.

SOURCE_SYSTEM = "football-data.org"

RAW_COMPETITIONS_ROOT = (
    f"{FOOTBALL_DATA_ORG_RAW_ROOT}/competitions"
)

BRONZE_TABLE = (
    f"{BRONZE_SCHEMA}."
    "football_data_org_competitions"
)

# -----------------------------------------------------------------------------
# Raw source filenames
# -----------------------------------------------------------------------------
# Raw filenames reflect the ingestion naming convention.
# Canonical league identifiers are assigned separately using
# FOOTBALL_DATA_ORG_COMPETITIONS.

RAW_COMPETITION_FILE_NAMES = {
    "PL": "pl_premier_league.json",
    "PD": "pd_la_liga.json",
    "BL1": "bl1_bundesliga.json",
    "SA": "sa_serie_a.json",
    "FL1": "fl1_ligue_1.json",
}

# -----------------------------------------------------------------------------
# Load football-data.org competition JSON files into Bronze
# -----------------------------------------------------------------------------
# Each raw file represents one competition.
#
# Example:
#   competitions/pl_premier_league.json
#
# The JSON payload contains nested source structures such as:
#
#   area
#   currentSeason
#   seasons
#
# Bronze intentionally preserves these nested source structures.
# Normalization, flattening, renaming, and analytical datatype decisions are
# deferred to the Silver layer.
#
# Technical metadata is added for lineage and traceability:
#
#   _league        - canonical project league identifier
#   _ingested_at   - timestamp of this Bronze load
#   _source_file   - Unity Catalog Volume file that produced the row
#   _source_system - originating external data source
#   _raw_sha256    - hash of the raw file used for change detection
#
# Each competition is maintained as one current Bronze snapshot.
# Unchanged files are skipped; changed files replace the previous row.

loaded_file_count = 0

for competition_code, league in FOOTBALL_DATA_ORG_COMPETITIONS.items():

    file_name = RAW_COMPETITION_FILE_NAMES[
        competition_code
    ]

    file_path = (
        f"{RAW_COMPETITIONS_ROOT}/"
        f"{file_name}"
    )

    # -------------------------------------------------------------------------
    # Incremental loading: skip unchanged competition metadata
    # -------------------------------------------------------------------------

    source_hash = calculate_file_sha256(file_path)

    replace_predicate = (
        f"_league = '{league}'"
    )

    if not bronze_file_needs_processing(
        spark=spark,
        table_name=BRONZE_TABLE,
        replace_predicate=replace_predicate,
        source_hash=source_hash,
    ):
        print(
            f"SKIP unchanged API competition: {file_path}"
        )
        continue

    source_df = (
        spark.read
        .option("multiLine", "true")
        .json(file_path)
    )

    bronze_df = (
        source_df
        .withColumn(
            "_league",
            F.lit(league),
        )
        .withColumn(
            "_ingested_at",
            F.current_timestamp(),
        )
        .withColumn(
            "_source_file",
            F.col("_metadata.file_path"),
        )
        .withColumn(
            "_source_system",
            F.lit(SOURCE_SYSTEM),
        )
    )

    (

    write_bronze_file_snapshot(
        df=bronze_df,
        table_name=BRONZE_TABLE,
        replace_predicate=replace_predicate,
        source_hash=source_hash,
    )

    )

    loaded_file_count += 1

    print(
        f"Loaded football-data.org Bronze file "
        f"{loaded_file_count}: "
        f"{competition_code} / {league} "
        f"-> {BRONZE_TABLE}"
    )


print(
    f"Successfully loaded {loaded_file_count} "
    "football-data.org competition source files into Bronze."
)